In [9]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,random_split
from torchvision import datasets,transforms
import torch.nn.functional as f
import torchvision.models as models
import time
import os
from matplotlib import pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [13]:
num_classes=6

In [8]:
transform = transforms.Compose([transforms.RandomHorizontalFlip(),
                                transforms.RandomRotation(10),
                                transforms.ColorJitter(brightness=0.2,contrast=0.2),
                                transforms.Resize((224,224)),
                                transforms.ToTensor(),
                                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

In [7]:
path="./dataset"
dataset=datasets.ImageFolder(root=path,transform=transform)
len(dataset)

2300

In [10]:
train_size=int(0.75*len(dataset))
val_size=len(dataset) - train_size
train_size,val_size

(1725, 575)

In [11]:
train_df,val_df= random_split(dataset,[train_size,val_size])

In [12]:
train_loader=DataLoader(train_df,batch_size=32,shuffle=True)
val_loader=DataLoader(val_df,batch_size=32)

In [31]:
class CarDamageClassifierCNN(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.network=nn.Sequential(
            nn.Conv2d(3,16,kernel_size=3,padding=1,stride=1), #16,224,224
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2,padding=0), #16,112,112
            nn.Conv2d(16,32,kernel_size=3,padding=1,stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2,padding=0),#32,56,56
            nn.Conv2d(32,64,kernel_size=3,padding=1,stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2,padding=0),#64,28,28
            nn.Flatten(),
            nn.Linear(64*28*28,512),
            nn.ReLU(),
            nn.Linear(512,num_classes),
            #nn.ReLU(),
            #nn.Linear(1024,num_classes)
        )

    def forward(self,x):
        x=self.network(x)
        return x
            
        

In [32]:
model=CarDamageClassifierCNN(num_classes=num_classes).to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [33]:
def train_model(model,loss_fn,optimizer,epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss=0.0
        for images,labels in train_loader:
            images,labels = images.to(device),labels.to(device)

            optimizer.zero_grad()
            outputs=model(images)
            loss=loss_fn(outputs,labels)
            loss.backward()
            optimizer.step()

            running_loss = loss.item() * images.size(0)
        epoch_loss = running_loss/len(train_loader.dataset)
        print(f"Epoch [{epoch+1} / {epochs}], Avg Loss:{epoch_loss:.4f}")

        model.eval()
        correct=0
        total=0
        all_labels=[]
        all_predictions=[]
        with torch.no_grad():
            for images,labels in val_loader:
                images,labels=images.to(device),labels.to(device)

                outputs=model(images)
                val,predicted = torch.max(outputs.data,1)
                total+=labels.size(0)
                correct+=(predicted==labels).sum().item()
                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())
            print(f"Validation Accuracy : {100*correct/total:.2f}")

    return all_labels,all_predictions

In [34]:
train_model(model,loss_fn,optimizer,epochs=5)

Epoch [1 / 5], Avg Loss:0.0218
Validation Accuracy : 45.39
Epoch [2 / 5], Avg Loss:0.0160
Validation Accuracy : 50.96
Epoch [3 / 5], Avg Loss:0.0142
Validation Accuracy : 54.26
Epoch [4 / 5], Avg Loss:0.0078
Validation Accuracy : 52.87
Epoch [5 / 5], Avg Loss:0.0032
Validation Accuracy : 53.74


([4,
  0,
  3,
  0,
  0,
  1,
  2,
  5,
  0,
  2,
  0,
  4,
  2,
  4,
  5,
  0,
  1,
  0,
  5,
  0,
  2,
  0,
  0,
  4,
  4,
  0,
  5,
  2,
  4,
  0,
  0,
  5,
  2,
  1,
  4,
  3,
  4,
  5,
  3,
  3,
  0,
  5,
  2,
  2,
  2,
  4,
  1,
  2,
  0,
  2,
  3,
  2,
  2,
  3,
  0,
  0,
  5,
  0,
  1,
  5,
  0,
  5,
  0,
  2,
  0,
  1,
  1,
  3,
  1,
  0,
  0,
  3,
  0,
  0,
  4,
  0,
  5,
  2,
  2,
  3,
  0,
  0,
  1,
  0,
  3,
  1,
  2,
  0,
  0,
  0,
  5,
  1,
  4,
  5,
  1,
  5,
  1,
  4,
  0,
  3,
  2,
  0,
  0,
  3,
  2,
  0,
  0,
  2,
  3,
  3,
  1,
  2,
  5,
  3,
  2,
  5,
  2,
  0,
  1,
  4,
  1,
  1,
  3,
  5,
  3,
  1,
  1,
  0,
  2,
  2,
  4,
  2,
  2,
  0,
  0,
  1,
  2,
  1,
  0,
  4,
  3,
  2,
  2,
  0,
  2,
  3,
  3,
  4,
  5,
  0,
  0,
  2,
  4,
  2,
  1,
  1,
  0,
  4,
  1,
  4,
  2,
  0,
  0,
  3,
  0,
  1,
  1,
  1,
  3,
  2,
  2,
  0,
  2,
  1,
  2,
  1,
  2,
  2,
  1,
  4,
  5,
  2,
  2,
  2,
  0,
  2,
  2,
  2,
  2,
  2,
  5,
  2,
  2,
  0,
  2,
  0,
  0,
  3,
  2,
  1,


In [35]:
class CarClassifierCNNWithRegularization(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1), # (16, 224, 224) 
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0), # (16, 112, 112),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1), 
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0), # (32, 56, 56)           
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1), 
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=0), # (64, 28, 28),
            nn.Flatten(),
            nn.Linear(64*28*28, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        x = self.network(x)
        return x

In [36]:
model = CarClassifierCNNWithRegularization(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

all_labels, all_predictions = train_model(model, criterion, optimizer,  epochs=10)

Epoch [1 / 10], Avg Loss:0.0211
Validation Accuracy : 46.09
Epoch [2 / 10], Avg Loss:0.0221
Validation Accuracy : 48.70
Epoch [3 / 10], Avg Loss:0.0207
Validation Accuracy : 55.13
Epoch [4 / 10], Avg Loss:0.0222
Validation Accuracy : 52.87
Epoch [5 / 10], Avg Loss:0.0202
Validation Accuracy : 50.43
Epoch [6 / 10], Avg Loss:0.0203
Validation Accuracy : 53.22
Epoch [7 / 10], Avg Loss:0.0201
Validation Accuracy : 55.83
Epoch [8 / 10], Avg Loss:0.0136
Validation Accuracy : 50.78
Epoch [9 / 10], Avg Loss:0.0190
Validation Accuracy : 52.87
Epoch [10 / 10], Avg Loss:0.0162
Validation Accuracy : 52.17


In [38]:
class CarClassifierResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.resnet50(weights='DEFAULT')
        for param in self.model.parameters():
            param.requires_grad = False
            
        for param in self.model.layer4.parameters():
            param.requires_grad = True            
            
        self.model.fc = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.model.fc.in_features, num_classes)
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [39]:
model = CarClassifierResNet(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.005)

labels, predictions = train_model(model, criterion, optimizer, epochs=10)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\mailr/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:07<00:00, 13.3MB/s]


Epoch [1 / 10], Avg Loss:0.0098
Validation Accuracy : 64.52
Epoch [2 / 10], Avg Loss:0.0043
Validation Accuracy : 79.83
Epoch [3 / 10], Avg Loss:0.0024
Validation Accuracy : 80.35
Epoch [4 / 10], Avg Loss:0.0011
Validation Accuracy : 79.13
Epoch [5 / 10], Avg Loss:0.0003
Validation Accuracy : 79.65
Epoch [6 / 10], Avg Loss:0.0010
Validation Accuracy : 79.83
Epoch [7 / 10], Avg Loss:0.0009
Validation Accuracy : 77.39
Epoch [8 / 10], Avg Loss:0.0011
Validation Accuracy : 77.57
Epoch [9 / 10], Avg Loss:0.0014
Validation Accuracy : 76.70
Epoch [10 / 10], Avg Loss:0.0022
Validation Accuracy : 81.04


In [40]:
torch.save(model.state_dict(), 'cnn_cardamage_model.pth')

In [1]:
from sklearn.metrics import classification_report

report = classification_report(labels, predictions)
print(report)

NameError: name 'labels' is not defined

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from matplotlib import pyplot as plt

conf_matrix = confusion_matrix(labels, predictions, labels=np.arange(num_classes))
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix for Vehicle Damage Classification")
plt.show()